In [1]:
!pip install selenium
!apt-get update
!apt install chromium-chromedriver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 28.8 MB/s eta 0:00:00
Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,2

In [2]:
import csv
import os
import time
from datetime import date, datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, NoSuchElementException
import shutil
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Inicialize o WebDriver
opções = webdriver.ChromeOptions()
opções.add_argument('--headless')
opções.add_argument('--no-sandbox')
opções.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=opções)

# Criar pasta de saída se não existir
output_folder = "output"
os.makedirs(output_folder, exist_ok=True)

# URL da categoria
urls = {
    "Pisos_revestimentos_e_porcelanatos": "https://www.joli.com.br/pisos-revestimentos-e-porcelanatos?page=1"
}

# Coleta
limite_coletas = 10

for categoria, url in urls.items():
    urls_produtos = set()
    pagina_atual = 1

    print(f"Iniciando a coleta de URLs da categoria: {categoria}")

    while len(urls_produtos) < limite_coletas:
        try:
            url_com_pagina = f"{url.split('?')[0]}?page={pagina_atual}"
            driver.get(url_com_pagina)
            time.sleep(3)

            produtos = WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.vtex-search-result-3-x-galleryItem a'))
            )

            for produto in produtos:
                try:
                    link = produto.get_attribute('href')
                    if link and link not in urls_produtos:
                        urls_produtos.add(link)
                        print(f"Coletado: {link}")
                        if len(urls_produtos) >= limite_coletas:
                            break
                except StaleElementReferenceException:
                    continue

            pagina_atual += 1

        except TimeoutException:
            print("Timeout ao carregar os produtos.")
            break

    # Salvar URLs em CSV
    nome_arquivo = os.path.join(output_folder, f'urls_{categoria}.csv')
    with open(nome_arquivo, 'w', newline='', encoding='utf-8') as arquivo_csv:
        escritor_csv = csv.writer(arquivo_csv)
        escritor_csv.writerow(['URL do Produto'])
        for link in urls_produtos:
            escritor_csv.writerow([link])

    print(f"URLs salvas em: {nome_arquivo}")

# Mover arquivos para o Google Drive
base_destination_dir = '/content/drive/MyDrive/WebScraping/Joli/WebScraping_coleta_Joli'
today = datetime.today().strftime('%Y-%m-%d')
destination_dir = os.path.join(base_destination_dir, today)
os.makedirs(destination_dir, exist_ok=True)

for file_name in os.listdir(output_folder):
    full_file_name = os.path.join(output_folder, file_name)
    if os.path.isfile(full_file_name):
        shutil.copy(full_file_name, destination_dir)

print(f"Arquivos movidos para: {destination_dir}")

# Fechar o driver
driver.quit()


Mounted at /content/drive
Iniciando a coleta de URLs da categoria: Pisos_revestimentos_e_porcelanatos
Coletado: https://www.joli.com.br/piso-61x61-ld-gales-hd-61-cx2-23m2--moderno-e-elegante-para-o-seu-ambiente-marcela/p
Coletado: https://www.joli.com.br/revestimento-34x60-pei3-lc-branco-cx2-10m2-proporciona-qualidade-e-estilo-para-o-seu-ambiente-formigres/p
Coletado: https://www.joli.com.br/piso-61x61-pei4-ld-cimentocolor-cinza-hd61-cx2-23m2-conforto-e-elegancia-para-seu-ambiente-formigres/p
Coletado: https://www.joli.com.br/piso-61x61-pei4-ld-tavola-mr-hd61-cx2-23m2-seu-espaco-moderno-e-sofisticado-formigres/p
Coletado: https://www.joli.com.br/piso-60x60-nevada-mate-rt60034-retificado-cx2-52m2-embramaco/p
Coletado: https://www.joli.com.br/piso-esmaltado-32x58-branco-tijolo-g40104-cx2-02m2-seu-ambiente-moderno-e-com-estilo-bellacer/p
Coletado: https://www.joli.com.br/piso-66x66-braga-mate-retificado-ldv2-cx2-18m2-formigres/p
Coletado: https://www.joli.com.br/revestimento-esmaltado-32x